In [11]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2 # pyright: ignore[reportMissingImports]
from tensorflow.keras.models import Model # pyright: ignore[reportMissingModuleSource]
from tensorflow.keras.layers import ( # type: ignore
    Dense, Dropout, GlobalAveragePooling2D, Input
)
from keras.optimizers.legacy import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.utils import plot_model
import warnings
warnings.filterwarnings('ignore')

#### LOADING PREPROCESSED DATA DONE IN PREPROCESSING STAGE

In [5]:
print("\n" + "=" * 60)
print("LOADING PREPROCESSED DATA")
print("=" * 60)
 
# Load numpy arrays
data = np.load("outputs/stage2_preprocessed_data.npz")
X_train = data['X_train']
y_train = data['y_train']
X_val = data['X_val']
y_val = data['y_val']
X_test = data['X_test']
y_test = data['y_test']
y_train_onehot = data['y_train_onehot']
y_val_onehot = data['y_val_onehot']
y_test_onehot = data['y_test_onehot']
 
# Load config
with open("outputs/stage2_config.json", 'r') as f:
    config = json.load(f)

# Reconstruct class weight dict with integer keys
class_weight_dict = {int(k): v for k, v in config['class_weight_dict'].items()}
label_to_idx = config['label_to_idx']
idx_to_label = {int(k): v for k, v in config['idx_to_label'].items()}
CLASS_NAMES = config['class_names']
IMG_SIZE = config['img_size']
BATCH_SIZE = config['batch_size']
NUM_CLASSES = config['num_classes']
 
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  Classes: {NUM_CLASSES}")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
 


LOADING PREPROCESSED DATA
  X_train: (6981, 224, 224, 3)
  X_val:   (1532, 224, 224, 3)
  X_test:  (1502, 224, 224, 3)
  Classes: 7
  Image size: 224x224


#### BUILD THE MODEL

In [6]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

print(f"\nBase model: MobileNetV2")
print(f"  Total layers:    {len(base_model.layers)}")
print(f"  Total params:    {base_model.count_params():,}")
print(f"  Trainable:       0 (all frozen)")
print(f"  Output shape:    {base_model.output_shape}")

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
x = base_model(inputs, training=False)  # training=False keeps BatchNorm in inference mode

x = GlobalAveragePooling2D(name='global_avg_pool')(x)
x = Dense(128, activation='relu', name='dense_128')(x)
x = Dropout(0.3, name='dropout_03')(x)
outputs = Dense(NUM_CLASSES, activation='softmax', name='output_7class')(x)
model = Model(inputs, outputs, name='SkinLesion_MobileNetV2')

9406464/9406464 [==============================] - 1s 0us/step

Base model: MobileNetV2
  Total layers:    154
  Total params:    2,257,984
  Trainable:       0 (all frozen)
  Output shape:    (None, 7, 7, 1280)


#### MODEL SUMMARY

In [9]:
model.summary()
 
# Count parameters
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params
 
print(f"\n  Total parameters:         {total_params:>10,}")
print(f"  Trainable parameters:     {trainable_params:>10,}")
print(f"  Non-trainable parameters: {non_trainable_params:>10,}")
print(f"  Trainable ratio:          {trainable_params/total_params*100:.2f}%")

Model: "SkinLesion_MobileNetV2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 224, 224, 3)]     0         
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_avg_pool (GlobalAve  (None, 1280)              0         
 ragePooling2D)                                                  
                                                                 
 dense_128 (Dense)           (None, 128)               163968    
                                                                 
 dropout_03 (Dropout)        (None, 128)               0         
                                                                 
 output_7class (Dense)       (None, 7)      

#### COMPILE THE MODEL

In [12]:
print("\n" + "=" * 60)
print("COMPILING MODEL")
print("=" * 60)
 
LEARNING_RATE = 0.0001
 
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
 
print(f"  Optimizer:      Adam")
print(f"  Learning rate:  {LEARNING_RATE}")
print(f"  Loss function:  Categorical Cross-Entropy")
print(f"  Metrics:        Accuracy")
 


COMPILING MODEL
  Optimizer:      Adam
  Learning rate:  0.0001
  Loss function:  Categorical Cross-Entropy
  Metrics:        Accuracy


#### SETUP CALLBACKS

In [13]:
print("\n" + "=" * 60)
print("TRAINING CALLBACKS")
print("=" * 60)
 
os.makedirs("models", exist_ok=True)
 
callbacks = [
    # Stop training if val_loss doesn't improve for 5 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
 
    # Reduce learning rate if val_loss plateaus for 3 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,         # Halve the learning rate
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
 
    # Save best model checkpoint
    ModelCheckpoint(
        filepath='models/mobilenetv2_skin_lesion_best.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]
 
print("  1. EarlyStopping:     patience=5, restore_best_weights=True")
print("  2. ReduceLROnPlateau: patience=3, factor=0.5, min_lr=1e-7")
print("  3. ModelCheckpoint:   saves best model to models/")


TRAINING CALLBACKS
  1. EarlyStopping:     patience=5, restore_best_weights=True
  2. ReduceLROnPlateau: patience=3, factor=0.5, min_lr=1e-7
  3. ModelCheckpoint:   saves best model to models/


#### SETUP DATA AUGMENTATION GENERATOR

In [14]:
from keras.preprocessing.image import ImageDataGenerator
 
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)
 
train_generator = train_datagen.flow(
    X_train, y_train_onehot,
    batch_size=BATCH_SIZE,
    shuffle=True
)
 
print(f"\n  Training with real-time augmentation")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Steps per epoch: {len(X_train) // BATCH_SIZE}")



  Training with real-time augmentation
  Batch size: 32
  Steps per epoch: 218


#### SAVE MODEL CONFIG 

In [18]:
stage3_config = {
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
    'base_model': 'MobileNetV2',
    'base_trainable': False,
    'total_params': int(total_params),
    'trainable_params': int(trainable_params),
    'callbacks': ['EarlyStopping(patience=5)', 'ReduceLROnPlateau(patience=3)', 'ModelCheckpoint'],
    'augmentation': {
        'rotation_range': 20,
        'width_shift_range': 0.1,
        'height_shift_range': 0.1,
        'horizontal_flip': True,
        'vertical_flip': True,
        'zoom_range': 0.1
    }
}
 
with open("outputs/stage3_config.json", 'w') as f:
    json.dump(stage3_config, f, indent=2)
 
print(f"\nSaved: outputs/stage3_config.json")


Saved: outputs/stage3_config.json
